In [1]:
import os, subprocess, tempfile, shutil
from fastapi import HTTPException

def render_mermaid_to_png(mermaid_code: str, timeout_sec: int = 30) -> bytes:
    with tempfile.TemporaryDirectory() as td:
        in_path = os.path.join(td, "diagram.mmd")
        out_path = os.path.join(td, "diagram.png")

        with open(in_path, "w", encoding="utf-8") as f:
            f.write(mermaid_code)

        mmdc = shutil.which("mmdc.cmd") or shutil.which("mmdc")
        if not mmdc:
            raise HTTPException(
                status_code=500,
                detail=(
                    "Mermaid CLI (mmdc) not found in PATH for this process. "
                    "In Jupyter, add npm global bin to PATH or run Jupyter from the same terminal where `mmdc -h` works."
                ),
            )

        try:
            proc = subprocess.run(
                [mmdc, "-i", in_path, "-o", out_path],
                capture_output=True,
                text=True,
                timeout=timeout_sec,
            )
        except subprocess.TimeoutExpired:
            raise HTTPException(status_code=504, detail="Render timeout (mmdc)")
        except FileNotFoundError as e:
            raise HTTPException(status_code=500, detail=f"Could not run mmdc: {e}")

        if proc.returncode != 0 or not os.path.exists(out_path):
            err = (proc.stderr or proc.stdout or "Unknown render error")[:1200]
            raise HTTPException(status_code=400, detail=f"Mermaid render failed: {err}")

        with open(out_path, "rb") as f:
            return f.read()

In [2]:
from fastapi import FastAPI, UploadFile, File, Query
from fastapi.responses import Response

app = FastAPI(title="UML Agentic Generator (Sequence PNG)")

def build_sequence_mermaid_stub(api_id: str) -> str:
    """
    Temporary: stub diagram (replace later with LangGraph+Groq builder).
    Keep it always valid so your render pipeline is stable.
    """
    return f"""sequenceDiagram
    autonumber
    participant U as User
    participant S as {api_id.upper()}_Service
    U->>S: Request
    S-->>U: Response
"""

@app.post("/uml/png")
async def generate_sequence_png(
    api_id: str = Query(..., pattern="^(api_1|api_2)$"),
    mode: str = Query(..., pattern="^(truth|perfect)$"),
    file: UploadFile = File(...),
):
    # For now we just read file to ensure upload pipeline works.
    _ = await file.read()

    # Later: use the file content to generate mermaid sequence diagram.
    mermaid_code = build_sequence_mermaid_stub(api_id)

    png_bytes = render_mermaid_to_png(mermaid_code)

    filename = f"{api_id}-{mode}-sequence.png"
    return Response(
        content=png_bytes,
        media_type="image/png",
        headers={"Content-Disposition": f'attachment; filename="{filename}"'},
    )

@app.get("/health")
def health():
    return {"status": "ok"}

In [3]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://127.0.0.1:8000")

Server started on http://127.0.0.1:8000


In [5]:
import os, shutil

npm_bin = r"C:\Users\Nishath_107520\AppData\Roaming\npm"
os.environ["PATH"] = npm_bin + ";" + os.environ["PATH"]

print("mmdc:", shutil.which("mmdc"))
print("mmdc.cmd:", shutil.which("mmdc.cmd"))

mmdc: C:\Users\Nishath_107520\AppData\Roaming\npm\mmdc.CMD
mmdc.cmd: C:\Users\Nishath_107520\AppData\Roaming\npm\mmdc.cmd


In [4]:
import shutil
print("mmdc:", shutil.which("mmdc"))
print("mmdc.cmd:", shutil.which("mmdc.cmd"))
print("java:", shutil.which("java"))
print("plantuml:", shutil.which("plantuml"))

mmdc: C:\Users\Nishath_107520\AppData\Roaming\npm\mmdc.CMD
mmdc.cmd: C:\Users\Nishath_107520\AppData\Roaming\npm\mmdc.cmd
java: C:\Program Files (x86)\Common Files\Oracle\Java\javapath\java.EXE
plantuml: None
